**Question answered:** Can we enrich raw events with product category and availability data to enable deeper analysis?

**Key finding:** Category hierarchy and availability status were successfully merged onto ~78% of events; ~22% of events have no category match due to missing item properties and are excluded from category-level analyses.

**Next:** `03_funnel_analysis.ipynb` — measuring the funnel and quantifying the revenue opportunity.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime



In [10]:
concat = pd.concat([
    pd.read_csv('../data/item_properties_part1.csv', usecols=['itemid', 'property', 'value', 'timestamp']), 
    pd.read_csv('../data/item_properties_part2.csv', usecols=['itemid', 'property', 'value', 'timestamp'])
])
cat_tree = pd.read_csv('../data/category_tree.csv')
events = pd.read_csv('../data/events.csv')
item_props_sorted = concat.sort_values(by = 'timestamp', ascending= True)
item_props_filtered  = item_props_sorted.drop_duplicates(['itemid', 'property'], keep = 'last')
item_props_filtered = item_props_filtered[item_props_filtered['property'].isin(['categoryid', 'available'])]

item_props_pivot = item_props_filtered.pivot(index= 'itemid', columns = 'property', values= 'value').reset_index()
item_props_pivot['categoryid'] = pd.to_numeric(item_props_pivot['categoryid'], errors='coerce')

props_category = pd.merge(
    item_props_pivot, 
    cat_tree, 
    on= 'categoryid', 
    how= 'left'
)

df = props_category[['itemid', 'categoryid', 'parentid', 'available']].merge(
    events, 
    on= 'itemid', 
    how = 'right'
)


In [11]:
df.to_csv('../data/processed/events_enriched.csv', index=False)
df.isnull().sum()

itemid                 0
categoryid        255585
parentid          255592
available         255585
timestamp              0
visitorid              0
event                  0
transactionid    2733644
dtype: int64

In [12]:
concat[concat['property'] == 'price']['property'].count()

np.int64(0)

In [13]:
df.shape()

TypeError: 'tuple' object is not callable

In [ ]:
concat['property'].value_counts()

property
888           3000398
790           1790516
available     1503639
categoryid     788214
6              631471
               ...   
782                 1
288                 1
722                 1
744                 1
769                 1
Name: count, Length: 1104, dtype: int64